# 02 - Classical Time-Series Models

Models evaluated:
- Naive
- Seasonal Naive
- Holt
- Holt-Winters (ETS)
- ARIMA (auto order by AIC)
- SARIMA (auto order by AIC)

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
DAILY_PATH = ARTIFACTS_DIR / 'daily_series.csv'

HORIZON = 30
N_SPLITS = 3
SEASONAL_PERIOD = 7

daily = pd.read_csv(DAILY_PATH, parse_dates=['date']).set_index('date')
daily.head()

,daily_revenue,daily_orders
date,,
2024-02-10,31740.30,122
2024-02-11,37996.92,141
2024-02-12,35297.50,139
2024-02-13,40122.00,150
2024-02-14,35806.81,152


In [5]:
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, np.abs(y_true))
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100


def metric_row(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAPE': safe_mape(y_true, y_pred)
    }


def build_rolling_splits(series, horizon=30, n_splits=3):
    first_train_end = len(series) - horizon * n_splits
    if first_train_end <= 35:
        raise ValueError('Not enough data for requested rolling setup.')
    splits = []
    for i in range(n_splits):
        train_end = first_train_end + i * horizon
        train = series.iloc[:train_end]
        test = series.iloc[train_end:train_end + horizon]
        splits.append((train, test))
    return splits


def find_best_arima_order(train):
    candidates = [(p, d, q) for p in [0, 1, 2, 3] for d in [0, 1] for q in [0, 1, 2, 3]]
    best = None
    best_aic = np.inf
    for order in candidates:
        try:
            fit = ARIMA(train, order=order).fit()
            if fit.aic < best_aic:
                best_aic = fit.aic
                best = order
        except Exception:
            pass
    return best if best is not None else (1, 1, 1)


def find_best_sarima_order(train, seasonal_period=7):
    pdq = [(p, d, q) for p in [0, 1, 2] for d in [0, 1] for q in [0, 1, 2]]
    seasonal_pdq = [
        (P, D, Q, seasonal_period)
        for P in [0, 1]
        for D in [0, 1]
        for Q in [0, 1]
    ]
    best = None
    best_aic = np.inf
    for order in pdq:
        for sorder in seasonal_pdq:
            try:
                fit = SARIMAX(
                    train,
                    order=order,
                    seasonal_order=sorder,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False)
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best = (order, sorder)
            except Exception:
                pass
    return best if best is not None else ((1, 1, 1), (1, 1, 1, seasonal_period))


def tune_classical_params(series, seasonal_period=7):
    # Tune once per target on historical full series, then reuse in CV and final fit.
    arima_order = find_best_arima_order(series)
    sarima_order, sarima_seasonal = find_best_sarima_order(series, seasonal_period=seasonal_period)
    return {
        'ARIMA': {'order': list(arima_order)},
        'SARIMA': {'order': list(sarima_order), 'seasonal_order': list(sarima_seasonal)}
    }

In [6]:
def forecast_classical(model_name, train, horizon, seasonal_period=7, tuned_params=None):
    idx = pd.date_range(train.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')

    if model_name == 'Naive':
        return pd.Series([train.iloc[-1]] * horizon, index=idx)

    if model_name == 'SeasonalNaive':
        base = train.iloc[-seasonal_period:].values
        vals = [base[i % seasonal_period] for i in range(horizon)]
        return pd.Series(vals, index=idx)

    if model_name == 'Holt':
        fit = ExponentialSmoothing(train, trend='add', seasonal=None).fit(optimized=True)
        return fit.forecast(horizon)

    if model_name == 'ETS':
        fit = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=seasonal_period).fit(optimized=True)
        return fit.forecast(horizon)

    if model_name == 'ARIMA':
        order = tuple(tuned_params.get('ARIMA', {}).get('order', [1, 1, 1])) if tuned_params else (1, 1, 1)
        fit = ARIMA(train, order=order).fit()
        return fit.forecast(horizon)

    if model_name == 'SARIMA':
        if tuned_params:
            order = tuple(tuned_params.get('SARIMA', {}).get('order', [1, 1, 1]))
            sorder = tuple(tuned_params.get('SARIMA', {}).get('seasonal_order', [1, 1, 1, seasonal_period]))
        else:
            order, sorder = (1, 1, 1), (1, 1, 1, seasonal_period)
        fit = SARIMAX(train, order=order, seasonal_order=sorder, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        return fit.forecast(horizon)

    raise ValueError(model_name)


models = ['Naive', 'SeasonalNaive', 'Holt', 'ETS', 'ARIMA', 'SARIMA']
targets = ['daily_revenue', 'daily_orders']

target_tuning = {}
all_rows = []

for target in targets:
    series = daily[target].astype(float)
    tuned_params = tune_classical_params(series, seasonal_period=SEASONAL_PERIOD)
    target_tuning[target] = tuned_params
    splits = build_rolling_splits(series, horizon=HORIZON, n_splits=N_SPLITS)

    for split_no, (train, test) in enumerate(splits, start=1):
        for model_name in models:
            try:
                pred = forecast_classical(
                    model_name,
                    train,
                    len(test),
                    seasonal_period=SEASONAL_PERIOD,
                    tuned_params=tuned_params
                )
                pred = pd.Series(pred.values, index=test.index)
                m = metric_row(test.values, pred.values)
                all_rows.append({
                    'target': target,
                    'split': split_no,
                    'model': model_name,
                    **m
                })
            except Exception as ex:
                all_rows.append({
                    'target': target,
                    'split': split_no,
                    'model': model_name,
                    'MAE': np.nan,
                    'MSE': np.nan,
                    'RMSE': np.nan,
                    'MAPE': np.nan,
                    'error': str(ex)
                })

classical_metrics = pd.DataFrame(all_rows)
summary = (
    classical_metrics
    .groupby(['target', 'model'], as_index=False)[['MAE', 'MSE', 'RMSE', 'MAPE']]
    .mean()
    .sort_values(['target', 'RMSE'])
)

classical_metrics.to_csv(ARTIFACTS_DIR / 'classical_cv_metrics.csv', index=False)
summary.to_csv(ARTIFACTS_DIR / 'classical_metrics_summary.csv', index=False)

best_classical = {}
for t in targets:
    best_classical[t] = summary[summary['target'] == t].iloc[0]['model']

with open(ARTIFACTS_DIR / 'best_classical_models.json', 'w', encoding='utf-8') as f:
    json.dump(best_classical, f, indent=2)

with open(ARTIFACTS_DIR / 'classical_tuning_params.json', 'w', encoding='utf-8') as f:
    json.dump(target_tuning, f, indent=2)

summary

c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RITIK SINHA\AppData\Local\P

,target,model,MAE,MSE,RMSE,MAPE
0,daily_orders,ARIMA,9.797719,1.526415e+02,12.290706,7.245702
4,daily_orders,SARIMA,9.862489,1.527575e+02,12.293453,7.307824
2,daily_orders,Holt,9.764759,1.535523e+02,12.330644,7.270014
1,daily_orders,ETS,9.998952,1.576251e+02,12.484049,7.447087
5,daily_orders,SeasonalNaive,16.266667,3.732222e+02,19.153298,12.139118
3,daily_orders,Naive,17.166667,4.955222e+02,20.209577,12.882922
6,daily_revenue,ARIMA,2794.837431,1.142162e+07,3364.485249,8.194805
7,daily_revenue,ETS,2807.969634,1.149766e+07,3375.226882,8.248748
8,daily_revenue,Holt,2774.002881,1.150476e+07,3382.551946,8.203084
10,daily_revenue,SARIMA,2846.281219,1.215581e+07,3471.637020,8.464170
